# 1. Setup & Database Connection

In [1]:
import duckdb
from pathlib import Path
import os

os.makedirs('../data/processed', exist_ok=True)

raw_dir = Path('../data/raw')
parquet_pattern = str(raw_dir / 'fhvhv_tripdata_2024-*.parquet')
zone_file = str(raw_dir / 'taxi_zone_lookup.csv')

con = duckdb.connect('../data/processed/nyc_taxi.db')

In [2]:
con.execute(f"""
    CREATE TABLE IF NOT EXISTS dim_zone AS
    SELECT * FROM read_csv_auto('{zone_file}')
""")

df_dim_zone_preview = con.execute("SELECT * FROM dim_zone LIMIT 5").df()
df_dim_zone_preview

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


# 2. Creating a dim base

In [3]:
con.execute(f"""
    CREATE OR REPLACE TABLE dim_base AS
    SELECT DISTINCT
        hvfhs_license_num,
        dispatching_base_num,
        CASE
            WHEN hvfhs_license_num = 'HV0003' THEN 'Uber'
            WHEN hvfhs_license_num = 'HV0005' THEN 'Lyft'
            ELSE 'Unknown'
        END AS platform_name
    FROM read_parquet('{parquet_pattern}')
""")

In [4]:

df_dim_base_preview = con.execute("SELECT * FROM dim_base LIMIT 10").df()
df_dim_base_preview

,hvfhs_license_num,dispatching_base_num,platform_name
0,HV0005,B03406,Lyft
1,HV0003,B03404,Uber


In [5]:
con.execute("SELECT COUNT(*) FROM dim_base").fetchone()[0]

2

# 3. Creating a dim time

In [6]:
con.execute(f"""
    CREATE OR REPLACE TABLE dim_time AS 
    SELECT DISTINCT 
        DATE_TRUNC('hour', pickup_datetime) AS time_id,
        DATE_TRUNC('day', pickup_datetime) AS day_id,
        EXTRACT(HOUR FROM pickup_datetime) AS hour_of_day,
        EXTRACT(DOW FROM pickup_datetime) AS day_of_week,
        CASE WHEN EXTRACT(DOW FROM pickup_datetime) IN (0, 6) THEN 1 ELSE 0 END AS is_weekend
        
    FROM read_parquet('{parquet_pattern}')
    WHERE pickup_datetime IS NOT NULL;
""")

In [7]:
df_dim_time_preview = con.execute("SELECT * FROM dim_time LIMIT 5").df()
df_dim_time_preview

,time_id,day_id,hour_of_day,day_of_week,is_weekend
0,2024-01-13 15:00:00,2024-01-13,15,6,1
1,2024-01-13 23:00:00,2024-01-13,23,6,1
2,2024-01-14 11:00:00,2024-01-14,11,0,1
3,2024-01-20 07:00:00,2024-01-20,7,6,1
4,2024-01-20 09:00:00,2024-01-20,9,6,1


# 4. Creating a fact trip

In [8]:
con.execute(f"""
    CREATE OR REPLACE TABLE fact_trip AS
    SELECT *
    FROM read_parquet('{parquet_pattern}')
    WHERE dropoff_datetime > pickup_datetime
      AND trip_miles > 0
      AND trip_time > 0
""")

total_fact_trip = con.execute("SELECT COUNT(*) FROM fact_trip").fetchone()[0]
print(f"Total baris fact_trip (sebelum deduplikasi): {total_fact_trip:,}")

Total baris fact_trip (sebelum deduplikasi): 239,426,737


In [9]:
con.execute("SET preserve_insertion_order = false")

con.execute(f"""
    CREATE OR REPLACE TABLE fact_trip_dedup AS
    SELECT * EXCLUDE (row_num)
    FROM (
        SELECT 
            *,
            ROW_NUMBER() OVER (
                PARTITION BY dispatching_base_num, pickup_datetime, dropoff_datetime, PULocationID, DOLocationID
                ORDER BY request_datetime
            ) AS row_num
        FROM fact_trip
    )
    WHERE row_num = 1
""")

total_dedup = con.execute("SELECT COUNT(*) FROM fact_trip_dedup").fetchone()[0]
print(f"Total baris setelah deduplikasi: {total_dedup:,}")

Total baris setelah deduplikasi: 239,426,298


In [10]:
con.execute("DROP TABLE fact_trip")
con.execute("ALTER TABLE fact_trip_dedup RENAME TO fact_trip")

total_final = con.execute("SELECT COUNT(*) FROM fact_trip").fetchone()[0]
print(f"Total baris fact_trip (final): {total_final:,}")

Total baris fact_trip (final): 239,426,298


# 5. SQL test

In [11]:
df_duplicate_check = con.execute(f"""
    WITH duplicate_group AS (
        SELECT
            dispatching_base_num,
            pickup_datetime,
            dropoff_datetime,
            PULocationID,
            DOLocationID,
            COUNT(*) AS duplicate_count
        FROM fact_trip
        GROUP BY
            dispatching_base_num,
            pickup_datetime,
            dropoff_datetime,
            PULocationID,
            DOLocationID
            HAVING COUNT(*) > 1
        )
        SELECT
            COUNT(*) AS double_combination_total,
            SUM(duplicate_count) AS duplicate_total_rows
        FROM duplicate_group
""").df()

df_duplicate_check

,double_combination_total,duplicate_total_rows
0,0,NaN


In [16]:
df_referential_integrity = con.execute(f"""
    SELECT 
        COUNT(*) FILTER (WHERE zone_pu.LocationID IS NULL) AS invalid_pickup_zone,
        COUNT(*) FILTER (WHERE zone_do.LocationID IS NULL) AS invalid_dropoff_zone
    FROM fact_trip AS trip
    LEFT JOIN dim_zone AS zone_pu
        ON trip.PULocationID = zone_pu.LocationID
    LEFT JOIN dim_zone AS zone_do
        ON trip.DOLocationID = zone_do.LocationID
""").df()

df_referential_integrity

,invalid_pickup_zone,invalid_dropoff_zone
0,0,0


In [18]:
df_no_negative_check = con.execute("""
    SELECT
        COUNT(*) FILTER (WHERE trip_miles <= 0) AS zero_or_negative_miles,
        COUNT(*) FILTER (WHERE trip_time <= 0) AS zero_or_negative_time,
    FROM fact_trip
""").df()

df_no_negative_check

,zero_or_negative_miles,zero_or_negative_time
0,0,0


In [20]:
con.close()